<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/11_Date_Functions_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

spark = SparkSession.builder \
    .appName("Topic11") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

data = [
    (1, "Raj",   "2024-01-15", "2024-03-20"),
    (2, "Priya", "2023-06-10", "2024-03-20"),
    (3, "Amit",  "2022-11-05", "2024-03-20"),
    (4, "Sara",  "2024-03-01", "2024-03-20"),
    (5, "Karan", None,         "2024-03-20")
]

columns = ["id", "name", "join_date", "today"]
df = spark.createDataFrame(data, columns)
df.show()

+---+-----+----------+----------+
| id| name| join_date|     today|
+---+-----+----------+----------+
|  1|  Raj|2024-01-15|2024-03-20|
|  2|Priya|2023-06-10|2024-03-20|
|  3| Amit|2022-11-05|2024-03-20|
|  4| Sara|2024-03-01|2024-03-20|
|  5|Karan|      NULL|2024-03-20|
+---+-----+----------+----------+



## **1. to_date — Convert String to Date**

In [2]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- join_date: string (nullable = true)
 |-- today: string (nullable = true)



In [8]:
df1 = df.withColumn("join_date", f.to_date(f.col("join_date"), "yyyy-MM-dd")).printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- join_date: date (nullable = true)
 |-- today: string (nullable = true)



In [4]:
""""
# Date with slashes
df.withColumn("date", F.to_date(F.col("join_date"), "yyyy/MM/dd"))

# Day first format
df.withColumn("date", F.to_date(F.col("join_date"), "dd-MM-yyyy"))

# Full month name
df.withColumn("date", F.to_date(F.col("join_date"), "MMMM dd, yyyy"))
# "January 15, 2024" → 2024-01-15 #dates could be in this format too. always check cols..
"""

'"\n# Date with slashes\ndf.withColumn("date", F.to_date(F.col("join_date"), "yyyy/MM/dd"))\n\n# Day first format\ndf.withColumn("date", F.to_date(F.col("join_date"), "dd-MM-yyyy"))\n\n# Full month name\ndf.withColumn("date", F.to_date(F.col("join_date"), "MMMM dd, yyyy"))\n# "January 15, 2024" → 2024-01-15 #dates could be in this format too. always check cols..\n'

## **2. current_date and current_timestamp**

In [5]:
df.withColumn("today", f.current_date()).show()

+---+-----+----------+----------+
| id| name| join_date|     today|
+---+-----+----------+----------+
|  1|  Raj|2024-01-15|2026-05-26|
|  2|Priya|2023-06-10|2026-05-26|
|  3| Amit|2022-11-05|2026-05-26|
|  4| Sara|2024-03-01|2026-05-26|
|  5|Karan|      NULL|2026-05-26|
+---+-----+----------+----------+



In [7]:
df.withColumn("timestamp", f.current_timestamp()).show(truncate=False)

+---+-----+----------+----------+--------------------------+
|id |name |join_date |today     |timestamp                 |
+---+-----+----------+----------+--------------------------+
|1  |Raj  |2024-01-15|2024-03-20|2026-05-26 10:33:23.742989|
|2  |Priya|2023-06-10|2024-03-20|2026-05-26 10:33:23.742989|
|3  |Amit |2022-11-05|2024-03-20|2026-05-26 10:33:23.742989|
|4  |Sara |2024-03-01|2024-03-20|2026-05-26 10:33:23.742989|
|5  |Karan|NULL      |2024-03-20|2026-05-26 10:33:23.742989|
+---+-----+----------+----------+--------------------------+



## **3. datediff — Days Between Two Dates**

In [13]:
df1  = df.withColumn("today", f.to_date(f.col("today"), "yyyy-MM-dd")).withColumn("join_date", f.to_date(f.col("join_date"), "yyyy-MM-dd"))
df1.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- join_date: date (nullable = true)
 |-- today: date (nullable = true)



In [14]:
df1.withColumn("days_diff", f.datediff(f.col("today"), f.col("join_date"))).show() #end>start

+---+-----+----------+----------+---------+
| id| name| join_date|     today|days_diff|
+---+-----+----------+----------+---------+
|  1|  Raj|2024-01-15|2024-03-20|       65|
|  2|Priya|2023-06-10|2024-03-20|      284|
|  3| Amit|2022-11-05|2024-03-20|      501|
|  4| Sara|2024-03-01|2024-03-20|       19|
|  5|Karan|      NULL|2024-03-20|     NULL|
+---+-----+----------+----------+---------+



## **4. date_format — Format Date for Display**

In [15]:
df1.withColumn("formatted_date", f.date_format(f.col("join_date"), "MMM dd, yyyy")).show()

+---+-----+----------+----------+--------------+
| id| name| join_date|     today|formatted_date|
+---+-----+----------+----------+--------------+
|  1|  Raj|2024-01-15|2024-03-20|  Jan 15, 2024|
|  2|Priya|2023-06-10|2024-03-20|  Jun 10, 2023|
|  3| Amit|2022-11-05|2024-03-20|  Nov 05, 2022|
|  4| Sara|2024-03-01|2024-03-20|  Mar 01, 2024|
|  5|Karan|      NULL|2024-03-20|          NULL|
+---+-----+----------+----------+--------------+



In [18]:
df1.withColumn("day", f.date_format(f.col("join_date"), "EEEE")).show()

+---+-----+----------+----------+--------+
| id| name| join_date|     today|     day|
+---+-----+----------+----------+--------+
|  1|  Raj|2024-01-15|2024-03-20|  Monday|
|  2|Priya|2023-06-10|2024-03-20|Saturday|
|  3| Amit|2022-11-05|2024-03-20|Saturday|
|  4| Sara|2024-03-01|2024-03-20|  Friday|
|  5|Karan|      NULL|2024-03-20|    NULL|
+---+-----+----------+----------+--------+



## **5. Extracting Parts of a Date**

In [22]:
df1.withColumn("year", f.year(f.col("join_date"))).withColumn("month", f.month(f.col("join_date"))).withColumn("day", f.dayofmonth(f.col("join_date"))).withColumn("quarter", f.quarter(f.col("join_date"))).withColumn("day_of_week", f.dayofweek(f.col("join_date"))).show()


+---+-----+----------+----------+----+-----+----+-------+-----------+
| id| name| join_date|     today|year|month| day|quarter|day_of_week|
+---+-----+----------+----------+----+-----+----+-------+-----------+
|  1|  Raj|2024-01-15|2024-03-20|2024|    1|  15|      1|          2|
|  2|Priya|2023-06-10|2024-03-20|2023|    6|  10|      2|          7|
|  3| Amit|2022-11-05|2024-03-20|2022|   11|   5|      4|          7|
|  4| Sara|2024-03-01|2024-03-20|2024|    3|   1|      1|          6|
|  5|Karan|      NULL|2024-03-20|NULL| NULL|NULL|   NULL|       NULL|
+---+-----+----------+----------+----+-----+----+-------+-----------+



In [20]:
#dayofweek returns 1=Sunday, 2=Monday ... 7=Saturday. Not 0-indexed like Python!

## **6. date_add / date_sub — Add or Subtract Days**

In [23]:
#add 90 days in join date

df1.withColumn("probation_end", f.date_add(f.col("join_date"), 90)).show()

+---+-----+----------+----------+-------------+
| id| name| join_date|     today|probation_end|
+---+-----+----------+----------+-------------+
|  1|  Raj|2024-01-15|2024-03-20|   2024-04-14|
|  2|Priya|2023-06-10|2024-03-20|   2023-09-08|
|  3| Amit|2022-11-05|2024-03-20|   2023-02-03|
|  4| Sara|2024-03-01|2024-03-20|   2024-05-30|
|  5|Karan|      NULL|2024-03-20|         NULL|
+---+-----+----------+----------+-------------+



In [24]:
#substract 7 days

df1.withColumn("week_before", f.date_sub(f.col("join_date"), 7)).show()

+---+-----+----------+----------+-----------+
| id| name| join_date|     today|week_before|
+---+-----+----------+----------+-----------+
|  1|  Raj|2024-01-15|2024-03-20| 2024-01-08|
|  2|Priya|2023-06-10|2024-03-20| 2023-06-03|
|  3| Amit|2022-11-05|2024-03-20| 2022-10-29|
|  4| Sara|2024-03-01|2024-03-20| 2024-02-23|
|  5|Karan|      NULL|2024-03-20|       NULL|
+---+-----+----------+----------+-----------+



## **7. months_between — Months Between Two Dates**

In [26]:
df1.withColumn("months_diff", f.round(f.months_between(f.col("today"), f.col("join_date")), 0)).show()

+---+-----+----------+----------+-----------+
| id| name| join_date|     today|months_diff|
+---+-----+----------+----------+-----------+
|  1|  Raj|2024-01-15|2024-03-20|        2.0|
|  2|Priya|2023-06-10|2024-03-20|        9.0|
|  3| Amit|2022-11-05|2024-03-20|       16.0|
|  4| Sara|2024-03-01|2024-03-20|        1.0|
|  5|Karan|      NULL|2024-03-20|       NULL|
+---+-----+----------+----------+-----------+



## **8. to_timestamp — For DateTime (Date + Time)**

In [28]:
data2 = [(1, "2026-01-15 10:30:00"),
         (2, "2026-02-20 14:45:30")]

df2 = spark.createDataFrame(data2, ["id", "event_time"])

df2.withColumn("event_time",
    f.to_timestamp(f.col("event_time"), "yyyy-MM-dd HH:mm:ss")
).printSchema()
# event_time: timestamp

root
 |-- id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)



# **Practice Task**

Using this data:

In [42]:
data = [
    (1, "Raj",   "15/01/2022"),
    (2, "Priya", "20/06/2023"),
    (3, "Amit",  "05/11/2021"),
    (4, "Sara",  "01/03/2024")
]
columns = ["id", "name", "join_date"]
df = spark.createDataFrame(data, columns)

Convert join_date from dd/MM/yyyy format to proper DateType

Calculate how many days each person has been employed (from join_date to today)

Calculate how many complete months employed (rounded to 1 decimal)

Extract the year and month of joining separately

Add a column five_year_mark showing the date exactly 5 years (1825 days) after joining



In [43]:
df = df.withColumn("join_date", f.to_date(f.col("join_date"), "dd/MM/yyyy"))
df.show()

+---+-----+----------+
| id| name| join_date|
+---+-----+----------+
|  1|  Raj|2022-01-15|
|  2|Priya|2023-06-20|
|  3| Amit|2021-11-05|
|  4| Sara|2024-03-01|
+---+-----+----------+



In [44]:
df = df.withColumn("days_employed", f.datediff(f.current_date(), f.col("join_date")))
df.show()

+---+-----+----------+-------------+
| id| name| join_date|days_employed|
+---+-----+----------+-------------+
|  1|  Raj|2022-01-15|         1592|
|  2|Priya|2023-06-20|         1071|
|  3| Amit|2021-11-05|         1663|
|  4| Sara|2024-03-01|          816|
+---+-----+----------+-------------+



In [47]:
df = df.withColumn("months_diff", f.round(f.months_between(f.current_date(), f.col("join_date")), 0))
df.show()


+---+-----+----------+-------------+-----------+
| id| name| join_date|days_employed|months_diff|
+---+-----+----------+-------------+-----------+
|  1|  Raj|2022-01-15|         1592|       52.0|
|  2|Priya|2023-06-20|         1071|       35.0|
|  3| Amit|2021-11-05|         1663|       55.0|
|  4| Sara|2024-03-01|          816|       27.0|
+---+-----+----------+-------------+-----------+



In [48]:
df.withColumn("year", f.year(f.col("join_date"))).withColumn("month", f.month(f.col("join_date"))).show()

+---+-----+----------+-------------+-----------+----+-----+
| id| name| join_date|days_employed|months_diff|year|month|
+---+-----+----------+-------------+-----------+----+-----+
|  1|  Raj|2022-01-15|         1592|       52.0|2022|    1|
|  2|Priya|2023-06-20|         1071|       35.0|2023|    6|
|  3| Amit|2021-11-05|         1663|       55.0|2021|   11|
|  4| Sara|2024-03-01|          816|       27.0|2024|    3|
+---+-----+----------+-------------+-----------+----+-----+



In [49]:
df = df.withColumn("five_year", f.date_add(f.col("join_date"), 1825))
df.show()

+---+-----+----------+-------------+-----------+----------+
| id| name| join_date|days_employed|months_diff| five_year|
+---+-----+----------+-------------+-----------+----------+
|  1|  Raj|2022-01-15|         1592|       52.0|2027-01-14|
|  2|Priya|2023-06-20|         1071|       35.0|2028-06-18|
|  3| Amit|2021-11-05|         1663|       55.0|2026-11-04|
|  4| Sara|2024-03-01|          816|       27.0|2029-02-28|
+---+-----+----------+-------------+-----------+----------+

